# 02 · Обучение агента

**Пайплайн НИР — стадия 2 из 3.** Обучение edit-агента (trim/extend, PPO) на
датасете из стадии 1.

1. чтение конфига обучения и датасета (стадия 1);
2. curriculum-обучение (тиры включаются по расписанию);
3. сохранение чекпоинта + визуализация истории;
4. eval по тирам.

Раньше: **`01_data_preparation.ipynb`**. Дальше: **`03_nbco_experiments.ipynb`**
использует обученный чекпоинт.

> Логика — в `training/` и `paper_experiments.training_lc`; ячейки только вызывают.

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Конфиг и основные настройки

Конфиг обучения читается из `cfg/train/edit_scratch*.yaml` (`load_train_config`).
Геометрию маршрутов (**min/max длина, число маршрутов**) можно переопределить —
оставь `None`, чтобы взять значение из YAML.

> Переопределённая геометрия должна совпадать с датасетом стадии 1 (иначе сначала
> перегенерируй данные в `01_data_preparation.ipynb` с теми же значениями).

In [ ]:
from connectpt.routes_generator.paper_experiments.training_lc import load_train_config

PROFILE = 'smoke'      # 'smoke' (1 итерация) | 'full' (полный прогон статьи)

# основные настройки маршрутов (None -> значение из конфига)
MIN_ROUTE_LEN = None   # напр. 8
MAX_ROUTE_LEN = None   # напр. 15
N_ROUTES      = None   # напр. 12

_ov = {'data.min_route_len': MIN_ROUTE_LEN, 'data.max_route_len': MAX_ROUTE_LEN,
       'data.target_n_routes': N_ROUTES}
overrides = [f'{k}={v}' for k, v in _ov.items() if v is not None]

name = 'edit_scratch_smoke' if PROFILE == 'smoke' else 'edit_scratch'
train_cfg = load_train_config(name, overrides=overrides)

print('конфиг      :', name, '| overrides:', overrides or '(нет)')
print('run.name    :', train_cfg.run.name)
print('итераций    :', train_cfg.train_loop.n_iterations)
print('датасет     :', train_cfg.data.dataset_dirname)
print('маршруты    :', f'{train_cfg.data.target_n_routes} шт, длина '
      f'{train_cfg.data.min_route_len}..{train_cfg.data.max_route_len}')
print('тиры        :', list(train_cfg.curriculum.tiers))
print('чекпоинт    :', train_cfg.paths.checkpoint_path)

## Чтение датасета и стратифицированный сплит

`TrainingDataModule` читает графы и seed-маршруты стадии 1; `stratified_split`
делит по тирам на train / val / monitor.

In [ ]:
from connectpt.routes_generator.training import TrainingDataModule
from connectpt.routes_generator.paper_experiments.training_lc import copytier_config

ds = copytier_config(train_cfg)
dm = TrainingDataModule(
    raw_graphs_path=ds.subset_pkl, lc_results_dir=ds.new_dataset_dir, device=device,
    min_route_len=ds.min_route_len, max_route_len=ds.max_route_len,
    target_n_routes=ds.target_n_routes).setup()
graphs, seed_routes = dm.graphs, dm.seed_routes
meta_df = pd.read_csv(ds.meta_csv)
tier_of = dict(zip(meta_df['graph_index'], meta_df['tier']))

(train_idx, val_idx, monitor_idx, train_by_tier, val_by_tier) = dm.stratified_split(
    tier_of, ds.tiers, train_fraction=float(train_cfg.data.train_fraction),
    n_val_per_tier=int(train_cfg.curriculum.n_val_per_tier),
    seed=int(train_cfg.data.split_seed))
print(f'графов={len(graphs)}  train={len(train_idx)}  val={len(val_idx)}  monitor={len(monitor_idx)}')

## Расписание curriculum

Тиры включаются постепенно: каждая строка — доля итераций, активные тиры, метка.

In [ ]:
n_iter = int(train_cfg.train_loop.n_iterations)
schedule = [(round(float(f) * n_iter), list(t), lab)
            for f, t, lab in train_cfg.curriculum.schedule]

def stage_spans():
    spans, prev = [], 0
    for until, _, label in schedule:
        spans.append((prev + 1, until, label)); prev = until
    return spans

display(pd.DataFrame([(a, b, lab) for a, b, lab in stage_spans()],
                     columns=['итерация с', 'по', 'активные тиры']))

## Обучение

`EditTrainingRun` гоняет PPO по curriculum и сохраняет лучший чекпоинт. Возвращает
историю (для графиков) и путь к чекпоинту.

In [ ]:
from connectpt.routes_generator.training import EditTrainingRun

artifact = EditTrainingRun(train_cfg).run()
history_df = artifact.history
best_model_path = artifact.checkpoint_path
print(f'обучение готово -> {best_model_path}  ({len(history_df)} эпох)')

## Визуализация истории обучения

`stitch_history` склеивает историю (при дообучении — с предыдущими прогонами),
`plot_training_history` рисует кривые с подсветкой стадий curriculum.

In [ ]:
from connectpt.routes_generator.paper_experiments.training_lc import (
    stitch_history, plot_training_history)

h, spans = stitch_history(cfg=train_cfg, history_df=history_df)
spans = spans or stage_spans()
_ = plot_training_history(h, spans, train_cfg)
plt.show()

## Оценка обученной политики по тирам

Пересобираем модель+cost из чекпоинта и считаем метрики по тирам
(ATT/RTT/CONN — минуты; d0/d1/d2/d_un — доли спроса по числу пересадок).

In [ ]:
from connectpt.routes_generator.paper_experiments.training_lc import (
    build_edit_model_and_cost, balanced_eval_by_tier, save_visual_examples,
    plot_balanced_examples, load_visual_examples)
from connectpt.routes_generator.core.checkpoints import CheckpointStore

_, cost_obj, model, _, ckpt_path = build_edit_model_and_cost(
    run_name=train_cfg.run.name, device=device,
    vary_weights=bool(train_cfg.cost.variable_weights),
    adj_weight=float(train_cfg.adjustment_degree_weight))
CheckpointStore.load_model_weights(model, ckpt_path, map_location=device)
model.eval()

eval_df, visual_examples = balanced_eval_by_tier(
    train_cfg, model=model, cost_obj=cost_obj, device=device,
    graphs=graphs, seed_routes=seed_routes, val_by_tier=val_by_tier)
display(eval_df)
save_visual_examples(train_cfg, visual_examples)

## Примеры «до / после» по тирам

In [ ]:
examples = load_visual_examples(train_cfg)
plot_balanced_examples(examples, graphs, list(train_cfg.curriculum.tiers))
plt.show()
print()
print('Обученный чекпоинт:', best_model_path)
print('Далее: 03_nbco_experiments.ipynb')